# Troubleshooting

Common problems and solutions.

## Problem 1: No Data Files Created

**Symptom**: `data/documents/` directory is empty

**Solution**:

In [1]:
# CORRECT: Subscribe DocumentLogger
from bluesky import RunEngine
from bluesky_config.callbacks import DocumentLogger

RE = RunEngine({})
doc_logger = DocumentLogger('data/documents')
RE.subscribe(doc_logger)  # ← CRITICAL! Don't forget this!

print("✓ DocumentLogger subscribed correctly")

ModuleNotFoundError: No module named 'bluesky_config'

## Problem 2: Cannot Retrieve from DataBroker

**Symptom**: `db[-1]` raises `IndexError`

**Solution**:

In [ ]:
from databroker import Broker

db = Broker.named('temp')

# Check if any runs exist
runs = list(db.search())
print(f"Total runs in catalog: {len(runs)}")

if len(runs) == 0:
    print("\n❌ No runs found!")
    print("Make sure you subscribed db.v1.insert:")
    print("  RE.subscribe(db.v1.insert)")
else:
    print(f"\n✓ Found {len(runs)} runs")

## Problem 3: Import Errors

**Symptom**: `ModuleNotFoundError: No module named 'bluesky_config'`

**Solution**:

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

print(f"✓ Added {parent_dir} to sys.path")

# Now imports should work
try:
    from bluesky_config import devices
    print("✓ bluesky_config imports successfully")
except ImportError as e:
    print(f"❌ Still can't import: {e}")

## Problem 4: Missing Packages

**Symptom**: `ModuleNotFoundError: No module named 'bluesky'`

**Solution**: Install required packages

```bash
conda activate gnuradio
pip install bluesky ophyd databroker matplotlib pandas scipy openpyxl h5py
```

## Problem 5: Data Looks Wrong

**Symptom**: Missing columns or strange values

**Solution**: Check descriptor and events

In [ ]:
# Diagnostic check
try:
    run = db[-1]
    
    # Check descriptor
    desc = list(run.metadata['descriptors'])[0]
    print("Data keys:")
    for key in desc['data_keys'].keys():
        print(f"  - {key}")
    
    # Check table
    table = run.table()
    print(f"\nTable shape: {table.shape}")
    print(f"Columns: {list(table.columns)}")
    
    # Check exit status
    exit_status = run.metadata['stop'].get('exit_status', 'unknown')
    print(f"\nExit status: {exit_status}")
    
except Exception as e:
    print(f"Error during diagnostic: {e}")

## Quick Diagnostic Test

Run this to test your setup:

In [ ]:
from ophyd import Signal
from bluesky.plans import scan
from bluesky_config.devices import MockDetector

# Test setup
RE = RunEngine({})
doc_logger = DocumentLogger('../data/documents')
RE.subscribe(doc_logger)
db = Broker.named('temp')
RE.subscribe(db.v1.insert)

# Test devices
motor = Signal(name='motor', value=0)
det = MockDetector(name='det')

# Test run
print("Running diagnostic test...")
uid = RE(scan([det], motor, 0, 5, 5))

# Verify
run = db[-1]
table = run.table()

if len(table) == 5:
    print("\n✓ ALL TESTS PASSED!")
    print("  Your setup is working correctly.")
else:
    print(f"\n❌ Test failed: got {len(table)} points, expected 5")

## Getting More Help

If you're still stuck:

1. Check the main documentation:
   - `README.md` - Framework overview
   - `CLAUDE.md` - Development guidelines
   - `DATA_RETRIEVAL_AND_ANALYSIS_GUIDE.md` - Complete reference

2. Run test suite:
   ```bash
   python test_all_experiments.py
   ```

3. Check Bluesky documentation:
   - https://blueskyproject.io/

## Next Steps

→ **Chapter 8: [Next Steps](08_next_steps.ipynb)**